# Notebook 01 — Data Generation

Generates scaled-down 2D Navier-Stokes trajectories on a periodic torus,
following the setup in Tran et al. (2023) Section 5.1 ("TorusLi").

**Scaling decisions relative to the paper:**
- Paper: 1000 train / 200 val / 200 test trajectories at 64×64, up to 50 timesteps per trajectory
- Ours:  800 train / 100 val / 100 test trajectories at 64×64, 20 timesteps per trajectory
- Paper trains for 100,000 steps; we will train for ~2000 steps (enough to see clear trends)

The dataset fits in memory (<100 MB total) and generates in ~5-10 minutes on a T4.

## Setup

If you're running on Colab, this notebook should live at `mini_project/notebooks/` inside your cloned fork. The cell below adjusts the path so the imports work.

In [ ]:
import sys
from pathlib import Path

# Point to the mini_project root (one level up from notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Generate the dataset

The generator uses a pseudo-spectral Navier-Stokes solver on the periodic torus with fixed viscosity and a fixed Kolmogorov-style forcing. Initial conditions are sampled from a Gaussian random field prior matching Li et al. (2021a).

In [ ]:
from data.generate_ns import generate_and_save

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = PROJECT_ROOT / 'ns_data'

meta = generate_and_save(
    out_dir=str(DATA_DIR),
    n_train=800, n_val=100, n_test=100,
    N=64, T=20,
    dt=1e-3, record_every=100,    # snapshot interval = 0.1 seconds
    viscosity=1e-3,
    batch_size=50,                 # how many trajectories to simulate in parallel
    device=device,
    seed=42,
)
print('metadata:', meta)

## Visualize a sample trajectory

Quick sanity check that the dynamics look like 2D turbulence — complex vortex structures evolving smoothly.

In [ ]:
train_data = torch.load(DATA_DIR / 'train.pt', weights_only=True)
print(f'train_data shape: {train_data.shape}')
print(f'vorticity range: [{train_data.min():.3f}, {train_data.max():.3f}]')
print(f'vorticity std: {train_data.std():.3f}')

In [ ]:
# Plot a few timesteps of trajectory 0
fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
timesteps = [0, 4, 9, 14, 19]
vmax = train_data[0].abs().max().item()
for ax, t in zip(axes, timesteps):
    im = ax.imshow(train_data[0, t], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f't = {t * 0.1:.1f}s')
    ax.axis('off')
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
plt.suptitle('Sample Navier-Stokes trajectory (vorticity)', y=1.02)
plt.savefig(PROJECT_ROOT / 'results' / 'sample_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

## Energy/enstrophy diagnostics

Sanity check: for forced viscous NS, the kinetic energy and enstrophy should evolve smoothly (no blow-up, no immediate crash to zero).

In [ ]:
# Enstrophy = 0.5 * mean(vorticity^2). Compute for the first 10 trajectories
enstrophy = 0.5 * (train_data[:10] ** 2).mean(dim=(-1, -2))

plt.figure(figsize=(8, 4))
for i in range(10):
    plt.plot(torch.arange(20) * 0.1, enstrophy[i], alpha=0.6)
plt.xlabel('time (s)')
plt.ylabel('enstrophy')
plt.title('Enstrophy evolution for 10 sample trajectories')
plt.grid(alpha=0.3)
plt.savefig(PROJECT_ROOT / 'results' / 'enstrophy_diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nIf curves look smooth and bounded, the solver is stable. ✓')